# Taller 2: Automatización del pipeline ETL (Spotify + Grammys)
**Curso:** ETL (G01) · **Programa:** Ingeniería de Datos e Inteligencia Artificial

Pipeline en **Apache Airflow** que extrae dos fuentes (CSV de Spotify y tabla de Grammys en una base de datos), valida la calidad con **Pandera**, transforma, combina, carga el resultado en la base de datos, lo exporta a CSV y alimenta un **reporte estático leído desde la base de datos**.

```
EXTRAER → VALIDAR → TRANSFORMAR → COMBINAR → CARGAR → VISUALIZAR
```




# Flujo del Pipeline ETL — etl_spotify_grammys

```mermaid
flowchart TB

    %% =========================
    %% FUENTES DE DATOS
    %% =========================

    A["🎵 Spotify Tracks<br/><b>spotify_tracks.csv</b>"]
    B["🏆 Grammys<br/><b>Tabla en SQLite</b>"]

    %% =========================
    %% EXTRACCIÓN SPOTIFY
    %% =========================

    A --> C["📄 read_csv<br/><br/>Leer archivo CSV"]

    C --> D{"🛡️ validar_calidad<br/><br/>Pandera<br/>BranchPythonOperator"}

    %% =========================
    %% VALIDACIÓN
    %% =========================

    D -->|✅ PASA| E["⚙️ transform_csv<br/><br/>Transformaciones<br/>del dataset Spotify"]

    D -->|❌ NO PASA| F["🚨 cuarentena_spotify<br/><br/>Guardar registros<br/>con errores"]

    %% =========================
    %% EXTRACCIÓN GRAMMYS
    %% =========================

    B --> G["🗄️ read_db<br/><br/>Leer datos de Grammys<br/>desde SQLite"]

    G --> H["⚙️ transform_db<br/><br/>Transformaciones<br/>del dataset Grammys"]

    %% =========================
    %% MERGE
    %% =========================

    E --> I["🔗 merge<br/><br/>Unir Spotify + Grammys<br/><b>por artista</b>"]

    H --> I

    %% =========================
    %% CARGA
    %% =========================

    I --> J["🗄️ load<br/><br/>Cargar tabla final<br/><b>tracks_grammys</b><br/>en SQLite"]

    %% =========================
    %% ALMACENAMIENTO
    %% =========================

    J --> K["📁 store<br/><br/>Guardar dataset transformado<br/>como CSV local<br/><br/>☁️ Google Drive<br/>(si se configura)"]

    %% =========================
    %% FINAL
    %% =========================

    K --> L(["🏁 fin"])

    F --> L

    %% =========================
    %% REPORTE
    %% =========================

    L --> M["📊 Reporte estático / Dashboard<br/><br/>Consultas SQL sobre<br/><b>tracks_grammys</b><br/><br/>⚠️ NO sobre el CSV"]

    %% =========================
    %% ESTILOS
    %% =========================

    classDef source fill:#E8F5E9,stroke:#16A085,stroke-width:2px,color:#1B1B1B;
    classDef extract fill:#E8F0FE,stroke:#2878D4,stroke-width:2px,color:#1B1B1B;
    classDef validation fill:#FFF4D6,stroke:#F39C12,stroke-width:2px,color:#1B1B1B;
    classDef transform fill:#E8F8F5,stroke:#16A085,stroke-width:2px,color:#1B1B1B;
    classDef quarantine fill:#FDEDEC,stroke:#E74C3C,stroke-width:2px,color:#1B1B1B;
    classDef merge fill:#E8F0FE,stroke:#2878D4,stroke-width:3px,color:#1B1B1B;
    classDef database fill:#E0F7FA,stroke:#00A6A6,stroke-width:2px,color:#1B1B1B;
    classDef storage fill:#F3E8FF,stroke:#8E44AD,stroke-width:2px,color:#1B1B1B;
    classDef finish fill:#2878D4,stroke:#1557A0,stroke-width:2px,color:white;
    classDef report fill:#E8F0FE,stroke:#2878D4,stroke-width:3px,color:#1B1B1B;

    class A,B source;
    class C,G extract;
    class D validation;
    class E,H transform;
    class F quarantine;
    class I merge;
    class J database;
    class K storage;
    class L finish;
    class M report;

    %% =========================
    %% ESTILOS DE CONEXIONES
    %% =========================

    linkStyle 3 stroke:#16A085,stroke-width:3px;
    linkStyle 4 stroke:#E74C3C,stroke-width:3px;
    linkStyle 9 stroke:#2878D4,stroke-width:3px;


  

## 1. Entorno
Airflow 2.11.2 en un entorno aislado (Python 3.12) para no chocar con las librerías de Colab. Se instala también **Pandera** dentro de ese entorno, porque es allí donde se ejecutan las tareas.

In [1]:
import os, sys, shutil, subprocess

AIRFLOW_VENV = '/content/airflow_venv'
AIRFLOW_HOME = '/content/airflow'

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=True)
uv = shutil.which('uv') or os.path.join(os.path.dirname(sys.executable), 'uv')

airflow_python = os.path.join(AIRFLOW_VENV, 'bin', 'python')
if not os.path.exists(airflow_python):
    if os.path.exists(AIRFLOW_VENV):
        shutil.rmtree(AIRFLOW_VENV)
    subprocess.run([uv, 'venv', AIRFLOW_VENV, '--python', '3.12'], check=True)

subprocess.run([uv, 'pip', 'install', '--python', airflow_python, 'apache-airflow==2.11.2',
                '--constraint', 'https://raw.githubusercontent.com/apache/airflow/constraints-2.11.2/constraints-3.12.txt'], check=True)
subprocess.run([uv, 'pip', 'install', '--python', airflow_python, 'pandera', 'pandas'], check=True)

os.environ['PATH'] = os.path.join(AIRFLOW_VENV, 'bin') + os.pathsep + os.environ['PATH']
os.environ['VIRTUAL_ENV'] = AIRFLOW_VENV
os.environ['AIRFLOW_HOME'] = AIRFLOW_HOME
os.environ['AIRFLOW__CORE__LOAD_EXAMPLES'] = 'False'
os.environ['AIRFLOW__WEBSERVER__WEB_SERVER_PORT'] = '8081'

os.makedirs(f'{AIRFLOW_HOME}/dags', exist_ok=True)
!airflow db migrate > /dev/null 2>&1
!airflow version

CalledProcessError: Command '['c:\\Users\\Yurani\\Documents\\ETL\\WORSHOP_2\\Taller2_ETL_Spotify_Grammys_COMPLETO\\.venv\\Scripts\\uv.EXE', 'pip', 'install', '--python', '/content/airflow_venv\\bin\\python', 'apache-airflow==2.11.2', '--constraint', 'https://raw.githubusercontent.com/apache/airflow/constraints-2.11.2/constraints-3.12.txt']' returned non-zero exit status 2.

### Dependencias (`requirements.txt`)


In [ ]:
%%writefile /content/requirements.txt
# Dependencias del Taller 2 - ETL Spotify + Grammys
# Python 3.12. Airflow debe instalarse con su archivo de constraints oficial:
#   pip install -r requirements.txt \
#     --constraint https://raw.githubusercontent.com/apache/airflow/constraints-2.11.2/constraints-3.12.txt

apache-airflow==2.11.2   # orquestacion (DAG)
pandas                   # lectura/transformacion/merge (version fijada por constraints de Airflow)
pandera                  # validacion de calidad de datos
matplotlib               # reporte estatico
numpy

## 2. Datos de entrada
Descarga los dos datasets de Kaggle (enlaces en el enunciado) y súbelos aquí:
- **Spotify Tracks Dataset** (`dataset.csv`, Maharshi Pandya)
- **Grammy Awards** (Unanimad): sirve el `.csv` o el `database.sqlite` (tabla `grammys`)

Ejecuta la celda y selecciona ambos archivos. (Alternativa: monta Google Drive y ajusta `SPOTIFY_SRC` / `GRAMMY_SRC`.)

In [ ]:
from google.colab import files

BASE_DIR = '/content/etl_taller'
os.environ['ETL_BASE_DIR'] = BASE_DIR
for d in ('data', 'db', 'staging', 'output'):
    os.makedirs(f'{BASE_DIR}/{d}', exist_ok=True)

subidos = files.upload()   # selecciona los 2 archivos
nombres = list(subidos.keys())
print('Archivos recibidos:', nombres)

SPOTIFY_SRC = next(n for n in nombres if n.lower().endswith('.csv') and 'grammy' not in n.lower())
GRAMMY_SRC  = next(n for n in nombres if 'grammy' in n.lower() or n.lower().endswith(('.sqlite', '.db')))
print('Spotify :', SPOTIFY_SRC)
print('Grammys :', GRAMMY_SRC)

shutil.copy(SPOTIFY_SRC, f'{BASE_DIR}/data/spotify_tracks.csv')

## 3. Carga inicial de Grammys en la base de datos
Según el enunciado, Grammys debe estar **ya cargado en una base de datos** antes de que Airflow lo lea. Se usa SQLite (`etl_taller.db`), . Esta celda es un paso previo; la lectura la hace el DAG (`read_db`).

In [ ]:
import sqlite3
import pandas as pd

DB_PATH = f'{BASE_DIR}/db/etl_taller.db'

if GRAMMY_SRC.lower().endswith('.csv'):
    grammys = pd.read_csv(GRAMMY_SRC)
else:  # database.sqlite de Kaggle
    src = sqlite3.connect(GRAMMY_SRC)
    grammys = pd.read_sql_query('SELECT * FROM grammys', src)
    src.close()

conn = sqlite3.connect(DB_PATH)
grammys.to_sql('grammys', conn, if_exists='replace', index=False)
print('Filas en tabla grammys:', conn.execute('SELECT COUNT(*) FROM grammys').fetchone()[0])
conn.close()

grammys.head()

## 4. Definición del DAG
Un solo archivo con todas las tareas (`PythonOperator`, y `BranchPythonOperator` para la validación, como vimos en la sesión de branching).

**Decisiones de diseño**
- **Validación (Pandera) sobre Spotify**, justo después de leer el CSV. Reglas: tipos de dato, no nulos en `track_id`, `popularity` y `track_genre`, rangos (`popularity` 0–100, features de audio 0–1, `loudness` −60…5 dB, `tempo` ≥ 0, `key` −1…11, `mode` ∈ {0,1}, `time_signature` 0…5, `duration_ms` ≥ 0; el CSV real trae una fila con duración 0 que es la fila nula, por eso no se exige > 0).
- **Pasa / No pasa:** el `BranchPythonOperator` decide la ruta. Si **pasa** → `transform_csv`. Si **no pasa** → `cuarentena_spotify` (guarda el CSV en cuarentena y las filas fallidas en `dq_failure_cases.csv`) y las tareas `merge`, `load` y `store` quedan *skipped*. El resultado de cada validación queda registrado en la tabla `dq_log` de la base de datos.
- **Advertencias (no bloquean):** metadatos nulos (`artists`, `track_name`, `album_name`) y duplicados `(track_id, track_genre)`; se cuentan en `dq_log` y se corrigen en la transformación.
- **Transformación Spotify:** elimina columna índice, filas sin artista/nombre y duplicados; `explicit`→0/1; `duration_min`; `popularity_level` (baja ≤33, media ≤66, alta); `primary_artist`; claves de artista normalizadas (minúsculas, sin tildes ni signos).
- **Transformación Grammys:** descarta `img`, `published_at`, `updated_at` y filas sin `artist` (≈38 % del dataset, ver limitaciones); conserva solo premios ganados (`winner`; en este dataset **todas** las filas son ganadoras, por eso se habla de *premios* y no de nominaciones); separa las colaboraciones ("Jamie Foxx & T-Pain" → *jamie foxx*, *tpain*) y agrega **por artista**: nº de premios, primer/último año y nº de categorías.
- **Lógica de combinación (merge):** los tracks de Spotify traen varios artistas separados por `;`. Se separan (*explode*), se hace **LEFT JOIN** con Grammys por la clave normalizada del artista y se re-agrega por `(track_id, track_genre)` (suma de los premios de los artistas del track) y se agrega `primary_artist_grammy_awards` con los premios solo del artista principal (primer artista del track), usada en el ranking de artistas del reporte. Los tracks sin artista premiado quedan con 0 (`has_grammy_artist = 0`), así no se pierde ningún track de Spotify.
- **Carga:** tabla `tracks_grammys` (con verificación de conteo). **Store:** el CSV se genera leyendo la tabla ya cargada.

In [ ]:
%%writefile /content/airflow/dags/etl_spotify_grammys.py
"""
DAG: etl_spotify_grammys
Pipeline ETL: Spotify (CSV) + Grammys (SQLite) -> validar -> transformar -> combinar -> cargar -> CSV
"""
import os
import re
import shutil
import sqlite3
import unicodedata
from datetime import datetime, timedelta

import pandas as pd
from airflow import DAG
from airflow.operators.empty import EmptyOperator
from airflow.operators.python import BranchPythonOperator, PythonOperator

# ----------------------------------------------------------------------------
# Rutas (se pueden cambiar con variables de entorno)
# ----------------------------------------------------------------------------
BASE_DIR = os.environ.get("ETL_BASE_DIR", "/content/etl_taller")
SPOTIFY_CSV = f"{BASE_DIR}/data/spotify_tracks.csv"
DB_PATH = f"{BASE_DIR}/db/etl_taller.db"
STAGING = f"{BASE_DIR}/staging"
OUTPUT_DIR = f"{BASE_DIR}/output"
DRIVE_DIR = os.environ.get("ETL_DRIVE_DIR", "")  # si existe, tambien se copia el CSV final alli
FINAL_CSV = f"{OUTPUT_DIR}/spotify_grammys.csv"

for _d in (STAGING, OUTPUT_DIR):
    os.makedirs(_d, exist_ok=True)


# ----------------------------------------------------------------------------
# Utilidades
# ----------------------------------------------------------------------------
def normalizar_nombre(texto):
    """minusculas, sin tildes, solo letras/numeros/espacios (clave para el merge por artista)."""
    if texto is None or (isinstance(texto, float) and pd.isna(texto)):
        return None
    t = unicodedata.normalize("NFKD", str(texto)).encode("ascii", "ignore").decode("ascii")
    t = re.sub(r"[^a-z0-9 ]", "", t.lower())
    t = re.sub(r"\s+", " ", t).strip()
    return t or None


SEPARADORES = re.compile(
    r"\s*(?:&|/|;|,|\band\b|\bfeaturing\b|\bfeat\.?|\bwith\b|\bx\b)\s*", re.IGNORECASE
)


def claves_artista_grammy(artista):
    """Claves de cruce de un artista de Grammys: nombre completo + cada integrante de la colaboracion.
    Ej: 'Jamie Foxx & T-Pain' -> {'jamie foxx t pain'->'jamie foxx tpain', 'jamie foxx', 'tpain'}"""
    if artista is None or (isinstance(artista, float) and pd.isna(artista)):
        return []
    texto = re.sub(r"\(.*?\)", "", str(artista))  # quita parentesis: "(With The Nash Ramblers)"
    claves = {normalizar_nombre(texto)} | {normalizar_nombre(p) for p in SEPARADORES.split(texto)}
    return sorted(k for k in claves if k and len(k) >= 3)


def registrar_dq(dataset, estado, n_filas, n_fallas, detalle):
    """Guarda el resultado de la validacion en la tabla dq_log de la base de datos."""
    conn = sqlite3.connect(DB_PATH)
    conn.execute(
        """CREATE TABLE IF NOT EXISTS dq_log (
               run_ts TEXT, dataset TEXT, estado TEXT,
               n_filas INTEGER, n_fallas INTEGER, detalle TEXT)"""
    )
    conn.execute(
        "INSERT INTO dq_log VALUES (?,?,?,?,?,?)",
        (datetime.now().isoformat(timespec="seconds"), dataset, estado, n_filas, n_fallas, detalle),
    )
    conn.commit()
    conn.close()


# ----------------------------------------------------------------------------
# 1. EXTRACCION
# ----------------------------------------------------------------------------
def read_csv_spotify():
    df = pd.read_csv(SPOTIFY_CSV)
    df.to_csv(f"{STAGING}/spotify_raw.csv", index=False)
    print(f"[read_csv] Spotify leido: {df.shape[0]} filas x {df.shape[1]} columnas")


def read_db_grammys():
    conn = sqlite3.connect(DB_PATH)
    df = pd.read_sql_query("SELECT * FROM grammys", conn)
    conn.close()
    df.to_csv(f"{STAGING}/grammys_raw.csv", index=False)
    print(f"[read_db] Grammys leido desde la BD: {df.shape[0]} filas x {df.shape[1]} columnas")


# ----------------------------------------------------------------------------
# 2. VALIDACION DE CALIDAD (Pandera) -> Branch: pasa / no pasa
# ----------------------------------------------------------------------------
def construir_schema():
    import pandera.pandas as pa
    from pandera import Check

    def f01(nombre):
        return pa.Column(float, Check.in_range(0, 1), nullable=False, coerce=True)

    return pa.DataFrameSchema(
        {
            "track_id": pa.Column(str, nullable=False),
            "artists": pa.Column(str, nullable=True),      # nulos -> advertencia, se eliminan al transformar
            "album_name": pa.Column(str, nullable=True),
            "track_name": pa.Column(str, nullable=True),
            "popularity": pa.Column(int, Check.in_range(0, 100), nullable=False, coerce=True),
            "duration_ms": pa.Column(int, Check.ge(0), nullable=False, coerce=True),  # el dataset trae 1 fila con 0 (la fila nula)
            "explicit": pa.Column(bool, nullable=False, coerce=True),
            "danceability": f01("danceability"),
            "energy": f01("energy"),
            "speechiness": f01("speechiness"),
            "acousticness": f01("acousticness"),
            "instrumentalness": f01("instrumentalness"),
            "liveness": f01("liveness"),
            "valence": f01("valence"),
            "loudness": pa.Column(float, Check.in_range(-60, 5), nullable=False, coerce=True),
            "tempo": pa.Column(float, Check.ge(0), nullable=False, coerce=True),
            "key": pa.Column(int, Check.in_range(-1, 11), nullable=False, coerce=True),
            "mode": pa.Column(int, Check.isin([0, 1]), nullable=False, coerce=True),
            "time_signature": pa.Column(int, Check.in_range(0, 5), nullable=False, coerce=True),
            "track_genre": pa.Column(str, nullable=False),
        },
        strict=False,
    )


def validar_calidad_spotify():
    import pandera.pandas as pa

    df = pd.read_csv(f"{STAGING}/spotify_raw.csv")
    schema = construir_schema()

    # Advertencias (no bloquean el pipeline, se documentan)
    n_nulos_meta = int(df[["artists", "album_name", "track_name"]].isna().any(axis=1).sum())
    n_dup = int(df.duplicated(subset=["track_id", "track_genre"]).sum())
    advertencias = f"filas_con_metadatos_nulos={n_nulos_meta}; duplicados_track_id_genre={n_dup}"

    try:
        schema.validate(df, lazy=True)
        print("[validar] RESULTADO: DATOS VALIDOS -> continua el pipeline")
        print(f"[validar] Advertencias: {advertencias}")
        registrar_dq("spotify", "PASA", len(df), 0, advertencias)
        return "transform_csv"
    except pa.errors.SchemaErrors as exc:
        fallas = exc.failure_cases
        fallas.to_csv(f"{OUTPUT_DIR}/dq_failure_cases.csv", index=False)
        resumen = fallas.groupby(["column", "check"]).size().to_string()
        print("[validar] RESULTADO: DATOS NO VALIDOS -> cuarentena")
        print(resumen)
        registrar_dq("spotify", "NO PASA", len(df), len(fallas), advertencias)
        return "cuarentena_spotify"


def cuarentena_spotify():
    os.makedirs(f"{OUTPUT_DIR}/cuarentena", exist_ok=True)
    shutil.copy(f"{STAGING}/spotify_raw.csv", f"{OUTPUT_DIR}/cuarentena/spotify_cuarentena.csv")
    print("[ALERTA] Spotify no paso la validacion. Detalle en output/dq_failure_cases.csv")


# ----------------------------------------------------------------------------
# 3. TRANSFORMACION
# ----------------------------------------------------------------------------
def transform_csv_spotify():
    df = pd.read_csv(f"{STAGING}/spotify_raw.csv")
    n0 = len(df)

    df = df.drop(columns=[c for c in df.columns if c.startswith("Unnamed")])
    df = df.dropna(subset=["artists", "track_name"])
    df = df.drop_duplicates(subset=["track_id", "track_genre"]).copy()

    df["explicit"] = df["explicit"].astype(int)
    df["duration_min"] = (df["duration_ms"] / 60000).round(2)
    df["popularity_level"] = pd.cut(
        df["popularity"], bins=[-1, 33, 66, 100], labels=["baja", "media", "alta"]
    ).astype(str)
    df["primary_artist"] = df["artists"].str.split(";").str[0].str.strip()
    df["artist_keys"] = df["artists"].apply(
        lambda s: ";".join(k for k in (normalizar_nombre(a) for a in s.split(";")) if k)
    )

    df.to_csv(f"{STAGING}/spotify_clean.csv", index=False)
    print(f"[transform_csv] {n0} -> {len(df)} filas")


def transform_db_grammys():
    df = pd.read_csv(f"{STAGING}/grammys_raw.csv")
    n0 = len(df)

    df = df.drop(columns=[c for c in ["img", "published_at", "updated_at"] if c in df.columns])
    df = df.dropna(subset=["artist"])
    # Solo premios ganados (en este dataset todas las filas son winner=True)
    df = df[df["winner"].astype(str).str.lower().isin(["true", "1", "1.0"])].copy()
    n_premios = len(df)

    df["artist_key"] = df["artist"].apply(claves_artista_grammy)
    df = df.explode("artist_key").dropna(subset=["artist_key"])

    agg = df.groupby("artist_key", as_index=False).agg(
        grammy_awards=("category", "size"),
        first_grammy_year=("year", "min"),
        last_grammy_year=("year", "max"),
        grammy_categories=("category", "nunique"),
    )
    agg.to_csv(f"{STAGING}/grammys_clean.csv", index=False)
    print(f"[transform_db] {n0} filas -> {n_premios} premios con artista -> {len(agg)} claves de artista")


# ----------------------------------------------------------------------------
# 4. MERGE
# ----------------------------------------------------------------------------
def merge_datasets():
    sp = pd.read_csv(f"{STAGING}/spotify_clean.csv")
    gr = pd.read_csv(f"{STAGING}/grammys_clean.csv")

    ex = sp[["track_id", "track_genre", "artist_keys"]].copy()
    ex["artist_key"] = ex["artist_keys"].str.split(";")
    ex = ex.explode("artist_key").merge(gr, on="artist_key", how="left")
    ex["artistas_con_grammy"] = ex["grammy_awards"].notna().astype(int)

    agg = ex.groupby(["track_id", "track_genre"], as_index=False).agg(
        grammy_awards=("grammy_awards", "sum"),
        first_grammy_year=("first_grammy_year", "min"),
        last_grammy_year=("last_grammy_year", "max"),
        artistas_con_grammy=("artistas_con_grammy", "sum"),
    )

    # Premios del ARTISTA PRINCIPAL (primer artista del track), sin sumar colaboradores
    premios_por_clave = gr.set_index("artist_key")["grammy_awards"]
    sp["_clave_principal"] = sp["artist_keys"].str.split(";").str[0]
    sp["primary_artist_grammy_awards"] = sp["_clave_principal"].map(premios_por_clave)

    final = sp.merge(agg, on=["track_id", "track_genre"], how="left").drop(
        columns=["artist_keys", "_clave_principal"]
    )
    for c in ["grammy_awards", "artistas_con_grammy", "primary_artist_grammy_awards"]:
        final[c] = final[c].fillna(0).astype(int)
    final["has_grammy_artist"] = (final["grammy_awards"] > 0).astype(int)

    final.to_csv(f"{STAGING}/final.csv", index=False)
    print(f"[merge] dataset final: {len(final)} filas; con artista Grammy: {final['has_grammy_artist'].sum()}")


# ----------------------------------------------------------------------------
# 5. CARGA y 6. ALMACENAMIENTO
# ----------------------------------------------------------------------------
def load_to_db():
    df = pd.read_csv(f"{STAGING}/final.csv")
    conn = sqlite3.connect(DB_PATH)
    df.to_sql("tracks_grammys", conn, if_exists="replace", index=False, chunksize=5000)
    n = conn.execute("SELECT COUNT(*) FROM tracks_grammys").fetchone()[0]
    conn.close()
    assert n == len(df), f"Conteo distinto tras la carga: {n} vs {len(df)}"
    print(f"[load] tabla tracks_grammys cargada: {n} filas")


def store_csv():
    # El CSV se genera leyendo lo que quedo en la BD (garantiza consistencia con la carga)
    conn = sqlite3.connect(DB_PATH)
    df = pd.read_sql_query("SELECT * FROM tracks_grammys", conn)
    conn.close()
    df.to_csv(FINAL_CSV, index=False)
    print(f"[store] CSV local: {FINAL_CSV}")
    if DRIVE_DIR and os.path.isdir(DRIVE_DIR):
        shutil.copy(FINAL_CSV, DRIVE_DIR)
        print(f"[store] copia en Google Drive: {DRIVE_DIR}")


# ----------------------------------------------------------------------------
# DAG
# ----------------------------------------------------------------------------
default_args = {
    "owner": "estudiante",
    "depends_on_past": False,
    "retries": 1,
    "retry_delay": timedelta(seconds=30),
}

with DAG(
    "etl_spotify_grammys",
    default_args=default_args,
    description="ETL Spotify (CSV) + Grammys (BD): validar, transformar, combinar, cargar",
    schedule_interval=None,
    start_date=datetime(2024, 1, 1),
    catchup=False,
    tags=["taller2", "spotify", "grammys"],
) as dag:

    read_csv = PythonOperator(task_id="read_csv", python_callable=read_csv_spotify)
    read_db = PythonOperator(task_id="read_db", python_callable=read_db_grammys)

    validar = BranchPythonOperator(task_id="validar_calidad", python_callable=validar_calidad_spotify)
    cuarentena = PythonOperator(task_id="cuarentena_spotify", python_callable=cuarentena_spotify)

    transform_csv = PythonOperator(task_id="transform_csv", python_callable=transform_csv_spotify)
    transform_db = PythonOperator(task_id="transform_db", python_callable=transform_db_grammys)

    merge = PythonOperator(task_id="merge", python_callable=merge_datasets)
    load = PythonOperator(task_id="load", python_callable=load_to_db)
    store = PythonOperator(task_id="store", python_callable=store_csv)

    fin = EmptyOperator(task_id="fin", trigger_rule="none_failed_min_one_success")

    read_csv >> validar >> [transform_csv, cuarentena]
    read_db >> transform_db
    [transform_csv, transform_db] >> merge >> load >> store >> fin
    cuarentena >> fin

## 5. Verificar el DAG y ejecutar el pipeline completo

In [ ]:
!airflow dags list 2>/dev/null | grep etl_spotify_grammys
!airflow tasks list etl_spotify_grammys --tree 2>/dev/null

In [ ]:
# Ejecución completa del DAG (respeta dependencias y la rama de validación)
!airflow dags test etl_spotify_grammys 2024-01-01 2>&1 | grep -E "\[(read_csv|read_db|validar|transform_csv|transform_db|merge|load|store|ALERTA)|Marking|RESULTADO|Branch|Skipping"

### Evidencia de ejecución
Estado de cada tarea (tabla interna de Airflow) y registro de calidad de datos guardado en nuestra base.

In [ ]:
con = sqlite3.connect('/content/airflow/airflow.db')
display(pd.read_sql_query("""
    SELECT task_id, state, start_date, end_date
    FROM task_instance
    WHERE dag_id = 'etl_spotify_grammys'
    ORDER BY start_date
""", con))
con.close()

con = sqlite3.connect(DB_PATH)
display(pd.read_sql_query('SELECT * FROM dq_log', con))
con.close()

### Prueba del camino "NO PASA"
Se corrompe temporalmente el CSV (popularidad = 150, danceability negativa), se ejecuta el DAG y se restaura el original. Debe verse la ruta de **cuarentena**, `merge/load/store` en *skipped*, y un registro `NO PASA` en `dq_log`.

In [ ]:
csv_path = f'{BASE_DIR}/data/spotify_tracks.csv'
shutil.copy(csv_path, '/content/spotify_backup.csv')

df_bad = pd.read_csv(csv_path)
df_bad.loc[0, 'popularity'] = 150
df_bad.loc[1, 'danceability'] = -0.5
df_bad.to_csv(csv_path, index=False)

!airflow dags test etl_spotify_grammys 2024-01-02 2>&1 | grep -E "\[(validar|ALERTA)|RESULTADO|Branch|Skipping|Marking"

shutil.copy('/content/spotify_backup.csv', csv_path)   # restaurar
print('CSV original restaurado')

con = sqlite3.connect(DB_PATH)
display(pd.read_sql_query('SELECT * FROM dq_log', con))
display(pd.read_csv(f'{BASE_DIR}/output/dq_failure_cases.csv').head())
con.close()

Con el CSV restaurado se vuelve a ejecutar el pipeline para dejar la base de datos con el resultado correcto:

In [ ]:
!airflow dags test etl_spotify_grammys 2024-01-03 2>&1 | grep -E "RESULTADO|\[load|\[store"

### (Opcional) Interfaz web de Airflow
Para la captura del grafo (`read_db → transform_db`, `read_csv → validar → transform_csv`, `merge → load → store`) levanta la interfaz y haz *Trigger DAG*.

In [ ]:
import time, urllib.request, re
AIRFLOW_EXE = f'{AIRFLOW_VENV}/bin/airflow'

os.system("fuser -k 8081/tcp > /dev/null 2>&1; pkill -f cloudflared > /dev/null 2>&1")
subprocess.run([AIRFLOW_EXE, 'users', 'create', '-u', 'admin', '-p', 'admin', '-f', 'Est', '-l', 'Colab',
                '-r', 'Admin', '-e', 'a@a.com'], capture_output=True)
get_ipython().system_raw(f'{AIRFLOW_EXE} scheduler > /content/scheduler.log 2>&1 &')
get_ipython().system_raw(f'{AIRFLOW_EXE} webserver --port 8081 > /content/airflow.log 2>&1 &')

for _ in range(60):
    try:
        if urllib.request.urlopen('http://127.0.0.1:8081/health', timeout=2).status == 200:
            break
    except Exception:
        time.sleep(1)

os.system("wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared; chmod +x /content/cloudflared")
get_ipython().system_raw('/content/cloudflared tunnel --url http://localhost:8081 > /content/tunnel.log 2>&1 &')
time.sleep(6)
urls = re.findall(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', open('/content/tunnel.log').read())
print('ACCESO A AIRFLOW:', urls[0] if urls else 'no se generó el túnel', '| usuario: admin | clave: admin')

## 6. Reporte estático (fuente: la base de datos)
Todas las cifras y gráficos salen de **consultas SQL a la tabla `tracks_grammys`** en `etl_taller.db`. El CSV exportado no se usa aquí.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

con = sqlite3.connect(DB_PATH)
q = lambda sql: pd.read_sql_query(sql, con)

kpi = q('''
    SELECT COUNT(*) AS tracks,
           SUM(has_grammy_artist) AS tracks_grammy,
           ROUND(100.0*SUM(has_grammy_artist)/COUNT(*), 1) AS pct_grammy,
           ROUND(AVG(popularity), 1) AS pop_prom
    FROM tracks_grammys''').iloc[0]

pop_grupo = q('''
    SELECT CASE has_grammy_artist WHEN 1 THEN 'Artista con Grammy' ELSE 'Sin Grammy' END AS grupo,
           ROUND(AVG(popularity), 2) AS popularidad
    FROM tracks_grammys GROUP BY has_grammy_artist''')

top_artistas = q('''
    SELECT primary_artist, MAX(primary_artist_grammy_awards) AS premios, ROUND(AVG(popularity), 1) AS pop_prom
    FROM tracks_grammys WHERE primary_artist_grammy_awards > 0
    GROUP BY primary_artist ORDER BY premios DESC, pop_prom DESC LIMIT 10''')

genero = q('''
    SELECT track_genre, ROUND(100.0*SUM(has_grammy_artist)/COUNT(*), 1) AS pct_grammy
    FROM tracks_grammys GROUP BY track_genre ORDER BY pct_grammy DESC LIMIT 12''')

dist = q('SELECT popularity, has_grammy_artist FROM tracks_grammys')

audio = q('''
    SELECT has_grammy_artist AS g, AVG(danceability) AS danceability, AVG(energy) AS energy,
           AVG(valence) AS valence, AVG(acousticness) AS acousticness
    FROM tracks_grammys GROUP BY has_grammy_artist ORDER BY g''')

buckets = q('''
    SELECT CASE WHEN primary_artist_grammy_awards = 0 THEN '0'
                WHEN primary_artist_grammy_awards <= 5 THEN '1-5'
                WHEN primary_artist_grammy_awards <= 15 THEN '6-15'
                ELSE '16+' END AS premios,
           ROUND(AVG(popularity), 2) AS popularidad, COUNT(*) AS tracks
    FROM tracks_grammys GROUP BY 1''')
orden = ['0', '1-5', '6-15', '16+']
buckets = buckets.set_index('premios').reindex(orden).reset_index().dropna()
con.close()

AZUL, NARANJA = '#1f77b4', '#ff7f0e'
fig, ax = plt.subplots(2, 3, figsize=(20, 11))
fig.suptitle(
    f"Spotify × Grammys — {int(kpi.tracks):,} tracks | {kpi.pct_grammy}% de artistas con Grammy | popularidad prom. {kpi.pop_prom}",
    fontsize=16, fontweight='bold')

ax[0,0].bar(pop_grupo.grupo, pop_grupo.popularidad, color=[NARANJA, AZUL][:len(pop_grupo)])
ax[0,0].set_title('Popularidad promedio'); ax[0,0].set_ylabel('popularity (0-100)')
for i, v in enumerate(pop_grupo.popularidad): ax[0,0].text(i, v, f'{v}', ha='center', va='bottom')

ax[0,1].barh(top_artistas.primary_artist[::-1], top_artistas.premios[::-1], color=NARANJA)
ax[0,1].set_title('Top 10 artistas con más premios Grammy (presentes en Spotify)'); ax[0,1].set_xlabel('premios')

ax[0,2].barh(genero.track_genre[::-1], genero.pct_grammy[::-1], color=AZUL)
ax[0,2].set_title('% de tracks con artista Grammy por género (top 12)'); ax[0,2].set_xlabel('%')

for g, lab, c in [(0, 'Sin Grammy', NARANJA), (1, 'Con Grammy', AZUL)]:
    ax[1,0].hist(dist[dist.has_grammy_artist == g].popularity, bins=30, alpha=.55, density=True, label=lab, color=c)
ax[1,0].set_title('Distribución de popularidad'); ax[1,0].set_xlabel('popularity'); ax[1,0].legend()

feats = ['danceability', 'energy', 'valence', 'acousticness']
x = np.arange(len(feats)); w = .38
for i, (_, r) in enumerate(audio.iterrows()):
    ax[1,1].bar(x + (i - .5) * w, [r[f] for f in feats], w, label='Con Grammy' if r.g == 1 else 'Sin Grammy',
                color=AZUL if r.g == 1 else NARANJA)
ax[1,1].set_xticks(x); ax[1,1].set_xticklabels(feats); ax[1,1].set_title('Perfil de audio promedio'); ax[1,1].legend()

ax[1,2].bar(buckets.premios, buckets.popularidad, color=AZUL)
ax[1,2].set_title('Popularidad según premios Grammy del artista principal'); ax[1,2].set_xlabel('premios'); ax[1,2].set_ylabel('popularidad prom.')
for i, (v, n) in enumerate(zip(buckets.popularidad, buckets.tracks)): ax[1,2].text(i, v, f'{v}\n(n={n:,})', ha='center', va='bottom', fontsize=9)

plt.tight_layout(rect=[0, 0, 1, .95])
os.makedirs(f'{BASE_DIR}/output', exist_ok=True)
plt.savefig(f'{BASE_DIR}/output/reporte_estatico.png', dpi=130)
plt.show()

In [ ]:
# Reporte estático en HTML (imagen incrustada), generado a partir del gráfico anterior
import base64
png = base64.b64encode(open(f'{BASE_DIR}/output/reporte_estatico.png', 'rb').read()).decode()
html = f'''<html><head><meta charset="utf-8"><title>Reporte Spotify x Grammys</title></head>
<body style="font-family:sans-serif;max-width:1400px;margin:auto">
<h1>Reporte Spotify × Grammys</h1>
<p>Fuente: tabla <code>tracks_grammys</code> de la base de datos <code>etl_taller.db</code>.</p>
<img src="data:image/png;base64,{png}" style="width:100%"></body></html>'''
open(f'{BASE_DIR}/output/reporte_estatico.html', 'w', encoding='utf-8').write(html)
print('Reporte guardado en', f'{BASE_DIR}/output/reporte_estatico.html')

## 7. Guardar resultados (CSV final, base de datos y reporte)

In [ ]:
print(os.listdir(f'{BASE_DIR}/output'))

# Opcional: copiar a Google Drive
from google.colab import drive
drive.mount('/content/drive')
DEST = '/content/drive/My Drive/Taller2_ETL'
os.makedirs(DEST, exist_ok=True)
for f in ['spotify_grammys.csv', 'reporte_estatico.png', 'reporte_estatico.html']:
    shutil.copy(f'{BASE_DIR}/output/{f}', DEST)
shutil.copy(DB_PATH, DEST)
print('Archivos copiados a', DEST)

## 8. Documentación técnica (resumen)

| Etapa | Tarea Airflow | Qué hace |
|---|---|---|
| Extracción CSV | `read_csv` | Lee `spotify_tracks.csv` con pandas |
| Extracción BD | `read_db` | `SELECT * FROM grammys` desde SQLite |
| Calidad de datos | `validar_calidad` | Esquema Pandera (`lazy=True`); decide ruta PASA / NO PASA y registra en `dq_log` |
| Cuarentena | `cuarentena_spotify` | Si falla: guarda el CSV en cuarentena + `dq_failure_cases.csv`; se omite el resto |
| Transformación | `transform_csv`, `transform_db` | Limpieza, variables derivadas y agregación de Grammys por artista |
| Combinación | `merge` | Explode de artistas + LEFT JOIN por clave normalizada + re-agregación por track |
| Carga | `load` | Tabla `tracks_grammys` en SQLite con verificación de conteo |
| Almacenamiento | `store` | CSV `spotify_grammys.csv` (local y, si se configura, Drive) |

**Orden de ejecución:** instalar entorno → subir datasets → cargar Grammys a la BD → escribir el DAG → `airflow dags test` → (opcional) UI → reporte estático.

**Limitaciones y decisiones a tener en cuenta**
- El cruce es por **nombre de artista** (normalizado). Variantes de escritura ("P!nk" vs "Pink") o artistas con varios nombres en una misma celda de Grammys pueden no emparejar; en esos casos el track queda con 0 premios. Con los datos reales se emparejan ≈713 de ≈1.650 artistas de Grammys y ≈11 % de los tracks.
- Grammys trae `artist` vacío en varias categorías (donde el premio va a compositores o productores, campo `workers`); esas filas no entran en el cruce.
- Los conteos de premios son **por artista** (histórico total), no por canción: un track hereda las cifras de sus artistas.
- Un track puede repetirse en varios géneros; el grano del dataset final es `(track_id, track_genre)`.
- Para este taller se usa SQLite como base de datos; el código solo depende de `sqlite3`/`pandas.to_sql`, por lo que migrar a PostgreSQL implica cambiar la conexión.